In [1]:
import pandas as pd 
import numpy as np 
# import matplotlib.pyplot as plt
# from pyfaidx import Fasta
# import regex
from sklearn.model_selection import train_test_split
import os

%matplotlib inline
# pd.options.mode.chained_assignment = None 
np.random.seed = 42

In [ ]:
#To create negative instances/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_Intersected_Homer/ex_Oridb_homer_final.bed
bed_columns = ['chr', 'start', 'end']
positive_instances = pd.read_csv("/ex_Oridb_homer_final.bed",sep='\t',header=None, names=bed_columns)
positive_instances['label']=1
positive_instances

,chr,start,end,label
0,chr1,30781,31280,1
1,chr1,70170,70669,1
2,chr1,124264,124763,1
3,chr1,159903,160402,1
4,chr1,175911,176410,1
...,...,...,...,...
293,chr16,776744,777243,1
294,chr16,818960,819459,1
295,chr16,842595,843094,1
296,chr16,880682,881181,1


In [3]:


df = pd.read_csv("selected_acs_negatives.csv")

indices = (
    df["sequence_id"]
    .astype(str)
    .str.extract(r"seq_(\d+)", expand=False)
    .dropna()
    .astype(int)
    .tolist()
)

print(indices)

[5232, 4229, 5431, 1404, 5371, 4004, 1080, 4511, 314, 1822, 2091, 2054, 3567, 4582, 558, 5888, 5544, 4684, 3739, 2272, 5329, 1471, 4201, 848, 3544, 1413, 4571, 4573, 2535, 5898, 4613, 3491, 265, 2480, 4049, 5723, 4605, 3068, 159, 1474, 3532, 5359, 1182, 2687, 3591, 741, 142, 2436, 645, 3952, 2281, 3090, 4733, 5569, 4125, 3456, 3524, 3887, 3383, 3595, 100, 1815, 5315, 3076, 2763, 2636, 2178, 3461, 377, 3824, 4059, 3665, 1257, 2016, 5440, 2843, 712, 1235, 1140, 4017, 4738, 2481, 5406, 846, 2671, 906, 1317, 1475, 4009, 3425, 5197, 1516, 3199, 1615, 2801, 4422, 5285, 5349, 3926, 1558, 368, 2590, 4822, 2876, 3714, 2728, 4718, 4640, 3820, 4947, 2851, 5053, 4871, 1479, 792, 4415, 4956, 5189, 2222, 2776, 5014, 2744, 5331, 2928, 2293, 326, 5058, 574, 5906, 252, 130, 3257, 4672, 2261, 2210, 5178, 318, 2255, 3548, 4667, 5181, 5398, 3490, 742, 320, 1637, 1326, 2657, 2130, 3671, 609, 3646, 4135, 5632, 3832, 1716, 3628, 3839, 2452, 666, 4397, 2999, 3879, 5261, 5547, 3313, 4106, 1582, 4930, 5776, 515

In [15]:
candidates_df = pd.read_csv("candidates_ACS_Neg.tsv",sep='\t')
matched_rows = candidates_df.iloc[indices].copy()
import re

def chr_sort_key(value):
    text = str(value)
    m = re.match(r"^chr(\d+)$", text)
    if m:
        return (0, int(m.group(1)))
    return (1, text)

matched_rows = matched_rows.sort_values(
    by="chr",
    key=lambda col: col.map(chr_sort_key)
).reset_index(drop=True)

# matched_rows.to_csv("candidates_ACS_Neg_selected_rows.tsv", index=False)
matched_rows

,chr,start,end,label,seq
0,chr1,180239,180738,0,TAAATCAAAAGCAGACTGGGAAGTTCTGTCGTAGGGATTTTTTTTT...
1,chr1,62452,62951,0,CTTCATCTTGAATTGTATCACCAGCATTGGTGGTGTCCTAGCTGTG...
2,chr1,132650,133149,0,AGATGCAGAAAATGTGCGATTGTGTAATGAGTGGAGGTCCTCACAC...
3,chr1,229027,229526,0,GGTACCAAAGGTAATGCATCTACCTCCCGTTACTTTTCCGAATCAG...
4,chr2,458288,458787,0,ATTGTTATCTTTATCAAAGAGGGCAAAGGCTTCTTTGAATTCAGCA...
...,...,...,...,...,...
293,chr16,209151,209650,0,ACCGTGGTAAGAACTCATTGGAAACAATACTATTGTTATTATGTTA...
294,chr16,105431,105930,0,GCGTTGACATGTTTACAGAGAATAATAGAATGGACCCTTTCCTCCT...
295,chr16,481043,481542,0,TAAAGAATGGTACCGGCCTGTTGTAAGGCTAGAGGAAAACGTATTT...
296,chr16,718376,718875,0,TAAATGGATTTTATAGAGGGTGAAAAGGACCAAAGGCAAAAATGTG...


In [27]:
negatives_df = matched_rows.loc[:, ["chr", "start", "end", "label"]].copy()
negatives_df.to_csv("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_ACS_Neg_Scoring/selected_acs_negatives.bed", sep="\t", header=False, index=False)
negatives_df.head()

,chr,start,end,label
0,chr1,180239,180738,0
1,chr1,62452,62951,0
2,chr1,132650,133149,0
3,chr1,229027,229526,0
4,chr2,458288,458787,0


In [28]:
# full_dataset = positive_df.append(pd.DataFrame(negative_df, columns=negative_df.columns))
full_dataset = pd.concat([positive_instances, pd.DataFrame(negatives_df, columns=positive_instances.columns)], ignore_index=True)

full_dataset['chr'] = pd.Categorical(full_dataset['chr'], categories=[f'chr{i}' for i in range(1, 17)], ordered=True)

# Sort DataFrame by 'chr' and then by 'start'
full_dataset_sorted = full_dataset.sort_values(by=['chr', 'start'])
full_dataset_sorted.to_csv('/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_ACS_Neg_Scoring/full_dataset_sorted1.bed', sep='\t', header=False, index=False)
full_dataset_sorted.to_csv('/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_ACS_Neg_Scoring/full_dataset_sorted1.csv', index=False)
full_dataset_sorted 

,chr,start,end,label
0,chr1,30781,31280,1
299,chr1,62452,62951,0
1,chr1,70170,70669,1
2,chr1,124264,124763,1
300,chr1,132650,133149,0
...,...,...,...,...
588,chr16,817097,817596,0
294,chr16,818960,819459,1
295,chr16,842595,843094,1
296,chr16,880682,881181,1


In [29]:
def seq2kmer(seq, k):
        """
        Convert original sequence to kmers
        
        Arguments:
        seq -- str, original sequence.
        k -- int, kmer of length k specified.
        
        Returns:
        kmers -- str, kmers separated by space

        """
        kmer = [seq[x:x+k] for x in range(len(seq)+1-k)]
        kmers = " ".join(kmer)
        return kmers

In [30]:
from pyfaidx import Fasta

data_path_DNABER="/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_ACS_Neg_Scoring/DNABERT"
data_path_DNABER2="/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_ACS_Neg_Scoring/DNABERT2"
window_dataset = pd.read_csv("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_ACS_Neg_Scoring/full_dataset_sorted1.bed", sep="\t", header=None, names=["chr","start","end", "label"])



sequence_data = Fasta(f"/p/project1/hai_dnaori/sgd_data/S288C_reference_sequence_R64-3-1_20210421.fsa")

# add the DNA sequence to each region
window_dataset["seq"] = window_dataset.apply(lambda x: sequence_data[x.chr][x.start-1: x.end].seq, axis=1)
#add kmer
window_dataset["kmer"] = window_dataset.apply(lambda x: seq2kmer(x.seq, 4), axis=1)
window_dataset


,chr,start,end,label,seq,kmer
0,chr1,30781,31280,1,ATATTTTAATGTTAAGATGAAATTTAAGTGAGCTGGTAATATCAAG...,ATAT TATT ATTT TTTT TTTA TTAA TAAT AATG ATGT T...
1,chr1,62452,62951,0,CTTCATCTTGAATTGTATCACCAGCATTGGTGGTGTCCTAGCTGTG...,CTTC TTCA TCAT CATC ATCT TCTT CTTG TTGA TGAA G...
2,chr1,70170,70669,1,AAGTGAGAAGAAAAAAAAAGGAAAAAAAGGAATTGTCCTAATGAGC...,AAGT AGTG GTGA TGAG GAGA AGAA GAAG AAGA AGAA G...
3,chr1,124264,124763,1,ACATAAATAACGACAAATGGTTGGTTATTTGAAGGATTAATGATCA...,ACAT CATA ATAA TAAA AAAT AATA ATAA TAAC AACG A...
4,chr1,132650,133149,0,AGATGCAGAAAATGTGCGATTGTGTAATGAGTGGAGGTCCTCACAC...,AGAT GATG ATGC TGCA GCAG CAGA AGAA GAAA AAAA A...
...,...,...,...,...,...,...
591,chr16,817097,817596,0,TTATTATTCAGCTCTATATCCTTTTTCCACTGCGATTGGTACTCTG...,TTAT TATT ATTA TTAT TATT ATTC TTCA TCAG CAGC A...
592,chr16,818960,819459,1,CCGAAAATTTAAGCGAAATTAAGAATAAAGAAGAAAAGAAAGAATT...,CCGA CGAA GAAA AAAA AAAT AATT ATTT TTTA TTAA T...
593,chr16,842595,843094,1,TGGCTGAGAGGAATTCTAAGATTTCTAATGTGGAGAGGTATACTCT...,TGGC GGCT GCTG CTGA TGAG GAGA AGAG GAGG AGGA G...
594,chr16,880682,881181,1,GGATTGTCAAGACACTCCGGTATTACTCGAGCCCGTAATACAACAG...,GGAT GATT ATTG TTGT TGTC GTCA TCAA CAAG AAGA A...


In [ ]:
# generate 7 data splits using random_state= 42, 25, 30, 50, 60, 70, 100
# train test split
X_train, X_test, y_train, y_test = train_test_split(window_dataset[["chr", "start", "end", "seq", "kmer"]], window_dataset.label, test_size=0.2, random_state=100, stratify=window_dataset.label)

# train valid split
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.125, random_state=100, stratify=y_train)
train_new = pd.DataFrame({"sequence": X_train.seq, "label": y_train.values})
valid_new = pd.DataFrame({"sequence": X_valid.seq, "label": y_valid.values})
test_new = pd.DataFrame({"sequence": X_test.seq, "label": y_test.values})

train_new_kmer = pd.DataFrame({"sequence": X_train.kmer, "label": y_train.values})
valid_new_kmer = pd.DataFrame({"sequence": X_valid.kmer, "label": y_valid.values})
test_new_kmer = pd.DataFrame({"sequence": X_test.kmer, "label": y_test.values})

folder= "train_dev_test100"
# save final kmer files to tsv
train_new_kmer.to_csv(os.path.join(data_path_DNABER, folder, "4/train.tsv"), sep="\t", index=False)
valid_new_kmer.to_csv(os.path.join(data_path_DNABER, folder,"4/dev.tsv"), sep="\t", index=False)
test_new_kmer.to_csv(os.path.join(data_path_DNABER, folder,"4/test.tsv"), sep="\t", index=False)

# save final files to tsv
train_new.to_csv(os.path.join(data_path_DNABER,folder, "train.tsv"), sep="\t", index=False)
valid_new.to_csv(os.path.join(data_path_DNABER,folder, "dev.tsv"), sep="\t", index=False)
test_new.to_csv(os.path.join(data_path_DNABER,folder, "test.tsv"), sep="\t", index=False)

#DNABERT2
train_new.to_csv(os.path.join(data_path_DNABER2,folder, "train.csv"), index=False)
valid_new.to_csv(os.path.join(data_path_DNABER2,folder, "dev.csv"), index=False)
test_new.to_csv(os.path.join(data_path_DNABER2,folder, "test.csv"), index=False)





train_window = pd.DataFrame({"chr": X_train.chr, "start": X_train.start, "end": X_train.end,"seq": X_train.seq, "label": y_train.values})
valid_window = pd.DataFrame({"chr": X_valid.chr, "start": X_valid.start, "end": X_valid.end,"seq": X_valid.seq, "label": y_valid.values})
test_window = pd.DataFrame({"chr": X_test.chr, "start": X_test.start, "end": X_test.end,"seq": X_test.seq, "label": y_test.values})


# train_window.drop(columns=["seq"], inplace=True)
train_window.to_csv(os.path.join(data_path_DNABER,folder,  "train_window.tsv"), sep="\t", index=False)
valid_window.to_csv(os.path.join(data_path_DNABER,folder,  "dev_window.tsv"), sep="\t", index=False)
test_window.to_csv(os.path.join(data_path_DNABER,folder,  "test_window.tsv"), sep="\t", index=False)
# # 
train_window.drop(columns=["seq"], inplace=True)
valid_window.drop(columns=["seq"], inplace=True)
test_window.drop(columns=["seq"], inplace=True)

train_window.to_csv(os.path.join(data_path_DNABER,folder,  "train_window.bed"),sep='\t', header=False, index=False)
valid_window.to_csv(os.path.join(data_path_DNABER,folder,  "dev_window.bed"),sep='\t', header=False, index=False)
test_window.to_csv(os.path.join(data_path_DNABER,folder,  "test_window.bed"),sep='\t', header=False, index=False)
